In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("sesion11-bi-ml-distribuido")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3228834a-34bf-4cf1-b80b-bb462b8a3e7e;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central


	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 284ms :: artifacts dl 9ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apac

26/10/24 19:05:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
RUTA_EVENTOS_S9 = "../s09-observabilidad-pipeline/artifacts/atmos_eventos_observabilidad"

eventos = spark.read.parquet(RUTA_EVENTOS_S9)

print(f"Filas eventos (S9): {eventos.count():,}")

Filas eventos (S9): 180


In [3]:
kpi_eventos = eventos.agg(
    F.count("*").alias("total"),
    F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
    F.avg("latencyMs").alias("avgLatencyMs"),
).first()

print(f"Eventos procesados:   {kpi_eventos['total']}")
print(f"Ratio de validos:     {kpi_eventos['ratioValidos']:.4f}")
print(f"Latencia promedio:    {kpi_eventos['avgLatencyMs']:.2f} ms")

Eventos procesados:   180
Ratio de validos:     1.0000
Latencia promedio:    2527.33 ms


In [4]:
import urllib.request

metricas_flujo = {
    "lambda26_bi_eventos_total": kpi_eventos["total"],
    "lambda26_bi_eventos_validos_ratio": kpi_eventos["ratioValidos"],
    "lambda26_bi_latencia_promedio_ms": kpi_eventos["avgLatencyMs"],
}

lineas = []
for nombre, valor in metricas_flujo.items():
    lineas.append(f"# TYPE {nombre} gauge")
    lineas.append(f"{nombre} {valor}")
cuerpo = "\n".join(lineas) + "\n"

PUSHGATEWAY_FLUJO_URL = "http://pushgateway:9091/metrics/job/lambda26_bi_flujo/instance/s11_batch"
peticion = urllib.request.Request(PUSHGATEWAY_FLUJO_URL, data=cuerpo.encode("utf-8"), method="PUT")
with urllib.request.urlopen(peticion) as respuesta:
    print("Pushgateway respondio:", respuesta.status)

Pushgateway respondio: 200


In [5]:
import json

for nombre in metricas_flujo:
    url = f"http://prometheus:9090/api/v1/query?query={nombre}"
    resultado = json.load(urllib.request.urlopen(url))
    valores = resultado["data"]["result"]
    if valores:
        print(f"{nombre}: {valores[0]['value'][1]}")
    else:
        print(f"{nombre}: SIN DATOS (revisa 3.1)")

lambda26_bi_eventos_total: SIN DATOS (revisa 3.1)
lambda26_bi_eventos_validos_ratio: SIN DATOS (revisa 3.1)
lambda26_bi_latencia_promedio_ms: SIN DATOS (revisa 3.1)


In [6]:
import time
time.sleep(20)  # un scrape_interval (15 s) despues del PUT
for nombre in metricas_flujo:
    valores = json.load(urllib.request.urlopen(f"http://prometheus:9090/api/v1/query?query={nombre}"))["data"]["result"]
    print(f"{nombre}: {valores[0]['value'][1] if valores else 'SIN DATOS'}")

lambda26_bi_eventos_total: 180
lambda26_bi_eventos_validos_ratio: 1
lambda26_bi_latencia_promedio_ms: 2527.3333333333335


## Parte B - ML en vivo

Antes de seguir, levanta de nuevo el productor de `campo-electrico-eventos` (S11, 3.5):

```bash
cd uso-campo-electrico
docker compose up -d
docker exec -it lambda26-uso-campo-electrico python producer_campo_electrico.py
```

In [7]:
from pyspark.ml import PipelineModel

RUTA_MODELO_S10 = "../s10-series-tiempo-inferencia-streaming/artifacts/models/modelo_ce_lr"

modelo_cargado = PipelineModel.load(RUTA_MODELO_S10)
print("Modelo cargado desde:", RUTA_MODELO_S10)

Modelo cargado desde: ../s10-series-tiempo-inferencia-streaming/artifacts/models/modelo_ce_lr


In [8]:
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("estacionId", StringType())
    .add("CE", DoubleType())
    .add("CM", DoubleType())
    .add("TempOut", DoubleType())
    .add("OutHum", DoubleType())
    .add("WindSpeed", DoubleType())
    .add("Bar", DoubleType())
    .add("SolarRad", DoubleType())
    .add("UVIndex", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "campo-electrico-eventos")
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

eventos_stream = (
    crudo.select(F.col("value").cast("string").alias("json_str"))
    .select(F.from_json(F.col("json_str"), esquema_evento).alias("e"))
    .select("e.*")
)

predicciones_stream = modelo_cargado.transform(eventos_stream)

In [9]:
RUTA_PREDICCIONES_STREAM = "./artifacts/predicciones_stream"
CHECKPOINT_STREAM = "./artifacts/chk_predicciones_stream"

In [10]:
import urllib.request

PUSHGATEWAY_BASE = "http://pushgateway:9091/metrics/job/lambda26_ml_streaming/instance"

def publicar_prediccion_en_vivo(estacion_id, ce_real, prediccion):
    cuerpo = (
        "# TYPE lambda26_ml_ce_real gauge\n"
        f"lambda26_ml_ce_real {ce_real}\n"
        "# TYPE lambda26_ml_prediccion_siguiente gauge\n"
        f"lambda26_ml_prediccion_siguiente {prediccion}\n"
    )
    peticion = urllib.request.Request(f"{PUSHGATEWAY_BASE}/{estacion_id}", data=cuerpo.encode("utf-8"), method="PUT")
    urllib.request.urlopen(peticion).close()

def guardar_predicciones_microlote(df_microlote, id_lote):
    filas = df_microlote.count()
    if filas == 0:
        print(f"Micro-lote {id_lote}: sin filas")
        return
    df_microlote.select(
        F.lit(id_lote).alias("batchId"), "estacionId",
        (F.col("timestamp") / 1000).cast("timestamp").alias("fechaHora"), "CE",
        F.col("prediction").alias("prediccionSiguienteCE"),
    ).write.mode("append").parquet(RUTA_PREDICCIONES_STREAM)

    for fila in df_microlote.select("estacionId", "CE", "prediction").collect():
        publicar_prediccion_en_vivo(fila["estacionId"], fila["CE"], fila["prediction"])

    print(f"Micro-lote {id_lote}: {filas} predicciones guardadas y publicadas en vivo")

# Momento 1: arrancar la consulta
consulta_predicciones = (
    predicciones_stream.writeStream
    .foreachBatch(guardar_predicciones_microlote)
    .option("checkpointLocation", CHECKPOINT_STREAM)
    .start()
)

In [11]:
import json
import time

def consultar(nombre):
    url = f"http://prometheus:9090/api/v1/query?query={nombre}"
    resultado = json.load(urllib.request.urlopen(url))
    valores = resultado["data"]["result"]
    return valores[0]["value"][1] if valores else None

primera = consultar("lambda26_ml_ce_real")
print("Primera lectura:", primera)

time.sleep(20)

segunda = consultar("lambda26_ml_ce_real")
print("Segunda lectura (20s despues):", segunda)

Primera lectura: None


Micro-lote 0: sin filas


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


Micro-lote 1: 1 predicciones guardadas y publicadas en vivo


Micro-lote 2: 1 predicciones guardadas y publicadas en vivo


Micro-lote 3: 1 predicciones guardadas y publicadas en vivo


Micro-lote 4: 1 predicciones guardadas y publicadas en vivo


Micro-lote 5: 1 predicciones guardadas y publicadas en vivo


Micro-lote 6: 1 predicciones guardadas y publicadas en vivo


Segunda lectura (20s despues): -4.01


Con la consulta activa se arma el tablero de Grafana (3.11) antes de detenerla.


In [12]:
# La consulta sigue corriendo en segundo plano mientras se arma el tablero de Grafana (3.11)
time.sleep(300)
print("micro-lotes procesados hasta ahora:", len(consulta_predicciones.recentProgress))
print("ultima lectura en Prometheus:", consultar("lambda26_ml_ce_real"), "| prediccion:", consultar("lambda26_ml_prediccion_siguiente"))

Micro-lote 7: 1 predicciones guardadas y publicadas en vivo


Micro-lote 8: 1 predicciones guardadas y publicadas en vivo


Micro-lote 9: 1 predicciones guardadas y publicadas en vivo


Micro-lote 10: 1 predicciones guardadas y publicadas en vivo


Micro-lote 11: 1 predicciones guardadas y publicadas en vivo


...

Micro-lote 103: 1 predicciones guardadas y publicadas en vivo


Micro-lote 104: 1 predicciones guardadas y publicadas en vivo


Micro-lote 105: 1 predicciones guardadas y publicadas en vivo


Micro-lote 106: 1 predicciones guardadas y publicadas en vivo


micro-lotes procesados hasta ahora: 99
ultima lectura en Prometheus: -4.59 | prediccion: -4.545755373758704


### 3.10 (Opcional) Detener la consulta y auditar lo recolectado


In [13]:
# Momento 2: detener la consulta (cuando ya tengas suficientes minutos reales)
consulta_predicciones.stop()

In [14]:
predicciones_hoy = spark.read.parquet(RUTA_PREDICCIONES_STREAM)

actual_siguiente = predicciones_hoy.select(
    (F.col("batchId") - 1).alias("batchId"),
    "estacionId",
    F.col("CE").alias("CE_real_siguiente"),
)

comparado = (
    predicciones_hoy.alias("p")
    .join(actual_siguiente.alias("a"), ["batchId", "estacionId"], "inner")
    .select("p.batchId", "p.estacionId", "p.CE", "p.prediccionSiguienteCE", "a.CE_real_siguiente")
    .withColumn("errorAbsoluto", F.abs(F.col("prediccionSiguienteCE") - F.col("CE_real_siguiente")))
)
comparado.orderBy("batchId").show(truncate=False)

kpi_prediccion = comparado.agg(
    F.count("*").alias("totalComparado"),
    F.avg("errorAbsoluto").alias("maeProduccion"),
).first()

print(f"Predicciones comparables: {kpi_prediccion['totalComparado']}")
print(f"MAE en produccion:        {kpi_prediccion['maeProduccion']:.4f}")

+-------+--------------+-----+---------------------+-----------------+---------------------+
|batchId|estacionId    |CE   |prediccionSiguienteCE|CE_real_siguiente|errorAbsoluto        |
+-------+--------------+-----+---------------------+-----------------+---------------------+
|1      |estacion-ns-01|-3.98|-3.9508926639395803  |-3.95            |8.926639395800962E-4 |
|2      |estacion-ns-01|-3.95|-3.921695128509099   |-3.94            |0.018304871490900876 |
|3      |estacion-ns-01|-3.94|-3.9115439414389925  |-3.95            |0.038456058561007644 |
|4      |estacion-ns-01|-3.95|-3.921002238698895   |-4.01            |0.08899776130110482  |
|5      |estacion-ns-01|-4.01|-3.979519280564091   |-3.96            |0.019519280564090913 |
|6      |estacion-ns-01|-3.96|-3.930047077731577   |-3.99            |0.059952922268423414 |
|7      |estacion-ns-01|-3.99|-3.959405428443401   |-4.07            |0.11059457155659924  |
|8      |estacion-ns-01|-4.07|-4.037020925280756   |-4.12            |

Predicciones comparables: 105
MAE en produccion:        0.0489


Limpia el valor congelado del Pushgateway (2.3) antes de armar el tablero, para no confundir un valor viejo con uno en vivo.


In [15]:
peticion_borrar = urllib.request.Request(f"{PUSHGATEWAY_BASE}/estacion-ns-01", method="DELETE")
urllib.request.urlopen(peticion_borrar).close()
print("Grupo borrado del Pushgateway")

Grupo borrado del Pushgateway


### 3.11 Armar el tablero de Grafana

Con la consulta todavia corriendo (no la detengas, esa es la celda de 3.10, opcional y para despues):

1. Abre `http://localhost:43000`, entra con `admin`/`admin`.
2. **Dashboards** -> **New** -> **New dashboard** -> **Add visualization**.
3. Elige el *datasource* **Prometheus**.
4. Modo **Code**, pega `lambda26_ml_ce_real`.
5. Boton **+ Query**, agrega `lambda26_ml_prediccion_siguiente`.
6. Visualizacion **Time series**.
7. Rango **Last 5 minutes**, *refresh* **5s**.
8. Titulo del panel: `CE real vs. prediccion (en vivo)`.
9. (Opcional) **Options** -> **Legend** -> **Custom**: `CE real` / `Prediccion siguiente`.
10. **Save dashboard**, nombralo `S11 - BI/ML distribuido`. Agrega tambien los dos paneles *Stat* del KPI batch (3.4).

Detalle completo: guia S11, 3.11.